# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashmita-21/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content item for one client on one reporting date.

**Tables:** I will use `fact_content_daily_performance` for daily performance measurements, `dim_content` for content metadata, and `dim_clients` for client data availability.

**Time window:** I will develop and verify my queries using the March 2026 partition (`month=2026-03`).

**Target / proxy:** I want to rank content items for human review based on their observed search-performance patterns. A future decline label will be defined separately from the feature window.

**Deliberate exclusion:** I will exclude client IDs, content IDs, future information, and label-derived columns from model features to avoid leakage.

In [15]:
%pip -q install duckdb

In [16]:
import duckdb
from google.colab import userdata

# Read the token securely from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Authenticate without printing or saving the token
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{hf_token}')"
)

print("DuckDB connection ready!")

DuckDB connection ready!


In [17]:
march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

schema = con.sql(
    f"DESCRIBE SELECT * FROM read_parquet('{march_path}')"
).df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature fields
- `gsc_impressions`: search visibility.
- `gsc_clicks`: search clicks.
- `gsc_avg_position`: average search position.
- `ga4_sessions`: website sessions.
- `ga4_pageviews`: page views.

These are candidate features because they describe observed performance during the measurement period.

### Label
- `future_decline`: a label I plan to define using a separate future observation window. It is not available at prediction time.

### Context fields
- `report_date`: identifies the reporting date.
- `month`: identifies the data partition.
- `gsc_data_available`: indicates whether Search Console data is available.
- `ga4_data_available`: indicates whether Analytics data is available.

### Excluded fields
- `client_hash_id` and `content_hash_id`: identifiers, not performance features.
- Future measurements and `future_decline`: excluded from model inputs to prevent leakage.
- Availability flags: used to filter data, not as predictive features.

In [18]:
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_pageviews"
]

label_column = "future_decline"

context_columns = [
    "report_date",
    "month",
    "gsc_data_available",
    "ga4_data_available"
]

excluded_columns = [
    "client_hash_id",
    "content_hash_id",
    "future_decline"
]

print("Features:", feature_columns)
print("Label:", label_column)
print("Context:", context_columns)
print("Excluded:", excluded_columns)

Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_pageviews']
Label: future_decline
Context: ['report_date', 'month', 'gsc_data_available', 'ga4_data_available']
Excluded: ['client_hash_id', 'content_hash_id', 'future_decline']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I will verify the daily grain, March 2026 row count and date range, and the availability of Search Console and Analytics data. I will use `IS TRUE` so that missing availability values are not incorrectly counted as available.

In [19]:
q1 = f"""
WITH grouped AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_per_key
    FROM read_parquet('{march_path}')
    GROUP BY 1, 2, 3
)
SELECT
    SUM(rows_per_key) AS total_rows,
    COUNT(*) AS distinct_keys,
    COUNT(*) FILTER (WHERE rows_per_key > 1) AS duplicate_keys
FROM grouped
"""
display(con.sql(q1).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_keys,duplicate_keys
0,9841378.0,9841378,0


In [20]:
q2 = f"""
SELECT
    COUNT(*) AS march_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM read_parquet('{march_path}')
"""
display(con.sql(q2).df())

,march_rows,first_date,last_date,clients,content_items
0,9841378,2026-03-01,2026-03-31,55,331437


In [21]:
q3 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
        AND ga4_data_available IS TRUE
    ) AS both_available
FROM read_parquet('{march_path}')
"""
display(con.sql(q3).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available,both_available
0,9841378,3611061,413966,364347


### Feature frame and decision timing

I use March 2026 measurements as features and April 2026 Search Console impressions to define a future-decline proxy. The five features are impressions, clicks, average position, sessions, and page views.

These measurements are knowable at the end of March because they come only from the March reporting period. April measurements are used only to create the future label, never as model features. I require Search Console availability in both months to define the label fairly.

In [22]:
april_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-04/*.parquet"
)

feature_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_impressions END) AS gsc_impressions_30d,

        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_clicks END) AS gsc_clicks_30d,

        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_avg_position * gsc_impressions END)
        / NULLIF(
            SUM(CASE WHEN gsc_data_available IS TRUE
                     THEN gsc_impressions END), 0
        ) AS gsc_avg_position_30d,

        SUM(CASE WHEN ga4_data_available IS TRUE
                 THEN ga4_sessions END) AS ga4_sessions_30d,

        SUM(CASE WHEN ga4_data_available IS TRUE
                 THEN ga4_pageviews END) AS ga4_pageviews_30d,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS march_gsc_days

    FROM read_parquet('{march_path}')
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE
                 THEN gsc_impressions END) AS april_impressions,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS april_gsc_days
    FROM read_parquet('{april_path}')
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.gsc_impressions_30d,
    m.gsc_clicks_30d,
    m.gsc_avg_position_30d,
    m.ga4_sessions_30d,
    m.ga4_pageviews_30d,
    CASE WHEN a.april_impressions < m.gsc_impressions_30d
         THEN 1 ELSE 0 END AS future_decline

FROM march m
JOIN april a
  ON m.client_hash_id = a.client_hash_id
 AND m.content_hash_id = a.content_hash_id

WHERE m.march_gsc_days > 0
  AND a.april_gsc_days > 0
"""

feature_frame = con.sql(feature_sql).df()

print("Feature frame shape:", feature_frame.shape)
print("Future decline distribution:")
display(feature_frame["future_decline"].value_counts())
display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (158549, 8)
Future decline distribution:


,count
future_decline,
1,93779
0,64770


,client_hash_id,content_hash_id,gsc_impressions_30d,gsc_clicks_30d,gsc_avg_position_30d,ga4_sessions_30d,ga4_pageviews_30d,future_decline
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.450877,NaN,NaN,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.298246,NaN,NaN,0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,5.637584,4.0,4.0,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.906404,9.0,12.0,0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,3.950542,3.0,3.0,0


### Leakage trap and honest result

I deliberately included `future_decline` as an input feature to demonstrate target leakage. The leaky model achieved 100% accuracy because it was given the answer it was supposed to predict.

After removing the label from the input features, the honest model achieved 64.34% accuracy. The difference was 35.66 percentage points.

The honest score is the meaningful result because it uses only March 2026 measurements to predict the April 2026 outcome. This is a preliminary result and does not prove that refreshing a page will improve its performance.

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# Five honest features (March 2026 data only)
features = [
    "gsc_impressions_30d",
    "gsc_clicks_30d",
    "gsc_avg_position_30d",
    "ga4_sessions_30d",
    "ga4_pageviews_30d"
]

# Future outcome label (April 2026)
y = feature_frame["future_decline"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    feature_frame[features],
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 1. Honest model: only March features
honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=5, random_state=42)
)

honest_model.fit(X_train, y_train)

honest_score = accuracy_score(
    y_test,
    honest_model.predict(X_test)
)

# 2. Leaky model: accidentally include the future label
X_leaky = feature_frame[features + ["future_decline"]]

XL_train, XL_test, yL_train, yL_test = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leaky_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=3, random_state=42)
)

leaky_model.fit(XL_train, yL_train)

leaky_score = accuracy_score(
    yL_test,
    leaky_model.predict(XL_test)
)

# Display results
print("Honest model accuracy:", round(honest_score, 4))
print("Leaky model accuracy:", round(leaky_score, 4))
print("Difference:", round(leaky_score - honest_score, 4))

Honest model accuracy: 0.6418
Leaky model accuracy: 1.0
Difference: 0.3582


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The dataset has some limitations.

1. Historical data may be unbalanced across clients and content items.
2. Some early records may have Search Console data but not Analytics data.
3. Missing availability values mean that a measurement cannot automatically be treated as zero.
4. Search performance can change because of seasonality, competition, algorithm updates, and other external factors.
5. A future decline label is a proxy and does not prove that editing a page will improve its performance.

My results will therefore be used for decision support and human review, not as proof of causation or guaranteed improvement.

In [24]:
print("Named limitation: Historical data availability may differ across clients.")
print("Missing data is not automatically treated as zero.")
print("The analysis supports prioritization, not causal claims.")

Named limitation: Historical data availability may differ across clients.
Missing data is not automatically treated as zero.
The analysis supports prioritization, not causal claims.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submitted my repo URL on the card. Done.